# Population ColorMNIST demo

This notebook reproduces the **ColorMNIST data-generating process used in the paper experiments**, but changes the training regime: instead of fixing one observational dataset and repeatedly cycling through it, the nuisance and target objectives are trained from **fresh population samples**.

The DGP is

$$
X\sim \mathrm{Unif}(0,1),\qquad
A\mid X=x\sim \mathrm{Bernoulli}\!\left(\sigma\{w(x-1/2)\}\right),
$$

with digits $(1,6)$ representing the two treatment arms. The foreground colour is

$$
(R,G,B)=(X,0,1-X),
$$

on a black background, using exactly the smooth foreground mask from the research experiment. We use the MNIST **test split**, matching the original `t10k` IDX files.

### Why a separate population trainer?

The normal `DeconfoundingFM.fit(X,A,Y)` API deliberately represents the real-data finite-sample estimator. Here we instead use `PopulationFlowTrainer`, which is only for simulations:

- the conditional outcome nuisance sees a newly sampled observational minibatch at **every update**;
- the target sees fresh observed $A,Y$ and fresh source samples from $P(Y\mid A=a)$ at **every update**;
- conditional-flow samples are expensive because they require ODE integration, so they are cached in a **renewable X-context reservoir** and refreshed periodically. Each target minibatch still draws fresh $X$; the plugin draw uses the nearest cached context.

Thus there is no fixed observational training sample. The nearest-context nuisance reservoir is the only deliberate computational approximation to completely on-the-fly population training.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import replace
import copy
import os
import sys
import warnings

import numpy as np
import torch
import matplotlib.pyplot as plt

# Run directly from a fresh clone before editable installation.
_here = Path.cwd().resolve()
for _root in (_here, _here.parent):
    if (_root / "src" / "deconfoundingfm").exists():
        sys.path.insert(0, str(_root / "src"))
        break

from deconfoundingfm.datasets import ColorMNISTConfig, ColorMNISTPopulation
from deconfoundingfm.experimental import PopulationFlowTrainer, PopulationTargetConfig
from deconfoundingfm.nuisance.outcome import ConditionalFlowFM, ConditionalFlowFMConfig
from deconfoundingfm.core.target import DeconfoundingFlow, DeconfoundingFlowConfig
from deconfoundingfm.nn.velocity import UNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
else:
    warnings.warn(
        "This full population ColorMNIST demo is GPU-oriented. The code is CPU-compatible, "
        "but the configured training run will be slow without CUDA."
    )

print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Configuration

The defaults below are close to the main ColorMNIST experiment: $w=5$, U-Net width 32, learning rate $10^{-4}$, Gaussian-base nuisance flow, and a small plugin reservoir. The deconfounding source has `BASE_NOISE_STD=0.1`, matching the smoothed empirical-base configuration used in the main image sweep; set it to `0.0` for the literal observational source.

The expensive part of population target fitting is refreshing the plugin reservoir. `CONTEXT_RESERVOIR_SIZE` and `CONTEXT_REFRESH_STEPS` control that cost directly.

In [ ]:
# Set DECONFOUNDINGFM_CMNIST_SMOKE=1 to exercise the complete notebook quickly.
SMOKE = os.environ.get("DECONFOUNDINGFM_CMNIST_SMOKE", "0") == "1"

# DGP
DIGITS = (1, 6)
CONFOUNDING_W = 5.0

# Population nuisance training
PLUGIN_STEPS = 1 if SMOKE else 20_000
PLUGIN_BATCH_SIZE = 2 if SMOKE else 128
PLUGIN_LR = 1e-4
PLUGIN_ODE_STEPS = 1 if SMOKE else 50
PLUGIN_UNET_C = 2 if SMOKE else 32

# Population target training
TARGET_STEPS = 1 if SMOKE else 20_000
TARGET_BATCH_SIZE = 2 if SMOKE else 128
TARGET_LR = 1e-4
TARGET_ODE_STEPS = 1 if SMOKE else 50
TARGET_UNET_C = 2 if SMOKE else 32
BASE_NOISE_STD = 0.1

# Renewable plugin context reservoir used by target training
PLUGIN_RESERVOIR = 1 if SMOKE else 5
PLUGIN_BATCH = 1
CONTEXT_RESERVOIR_SIZE = 2 if SMOKE else 2_048
CONTEXT_REFRESH_STEPS = 1 if SMOKE else 1_000
PLUGIN_CONTEXT_CHUNK = 2 if SMOKE else 32
RESERVOIR_DTYPE = "float16" if DEVICE.type == "cuda" else "float32"

# Mixed precision is used for nuisance + independent/Gaussian target updates.
# OT/Sinkhorn target updates are intentionally kept in float32.
USE_AMP = DEVICE.type == "cuda" and not SMOKE

# Evaluation
N_EVAL = 8 if SMOKE else 1_500
N_PROJECTIONS = 4 if SMOKE else 64
EVAL_CHUNK = 4 if SMOKE else 128

print("smoke mode:", SMOKE)


## 2. Load the same MNIST shape population

The original experiment loaded the raw MNIST `t10k` files. `train=False` below is the same split. The first run downloads MNIST into `DATA_ROOT`; on a cluster without outbound internet, download it once beforehand or use `ColorMNISTPopulation.from_idx(...)` with local IDX files.

In [ ]:
DATA_ROOT = Path.home() / ".cache" / "deconfoundingfm" / "mnist"
IDX_DIR = os.environ.get("DECONFOUNDINGFM_MNIST_IDX_DIR")

source_config = ColorMNISTConfig(
    digits=DIGITS,
    confounding_w=CONFOUNDING_W,
    tau=0.08,
    mask_sharpness=10.0,
    fg_alpha=0.0,
)

if IDX_DIR:
    idx_root = Path(IDX_DIR)
    source = ColorMNISTPopulation.from_idx(
        idx_root / "t10k-images.idx3-ubyte",
        idx_root / "t10k-labels.idx1-ubyte",
        config=source_config,
        device=DEVICE,
    )
else:
    source = ColorMNISTPopulation.from_torchvision(
        DATA_ROOT,
        config=source_config,
        train=False,
        download=True,
        device=DEVICE,
    )

propensity = source.oracle_propensity()
trainer = PopulationFlowTrainer(source)

print("digit pools:", {d: tuple(pool.shape) for d, pool in source.digit_pools.items()})


In [ ]:
# Inspect the confounding mechanism and a fresh observational batch.
preview = source.sample_observational(50 if SMOKE else 1_500)
X_preview = preview["X"].squeeze().cpu()
A_preview = preview["A"].squeeze().cpu()

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(X_preview[A_preview == 0], bins=35, density=True, alpha=0.55, label="A=0")
ax.hist(X_preview[A_preview == 1], bins=35, density=True, alpha=0.55, label="A=1")
ax.axhline(1.0, linestyle="--", linewidth=1.5, label="population P(X)")
ax.set_xlabel("foreground colour X")
ax.set_ylabel("density")
ax.set_title(f"ColorMNIST confounding, w={CONFOUNDING_W:g}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## 3. Train the conditional outcome nuisance on fresh population minibatches

The nuisance is exactly the image conditional-FM architecture used by the experiment: a Gaussian-base `UNetX` conditioned on continuous $X$ and treatment $A$. Unlike the finite-data experiment, **no minibatch is revisited** here.

In [ ]:
plugin_cfg = ConditionalFlowFMConfig(
    dim_y=1,  # unused for image mode
    dim_x=1,
    lr=PLUGIN_LR,
    batch_size=PLUGIN_BATCH_SIZE,
    ode_steps=PLUGIN_ODE_STEPS,
    base_kind="gaussian",
    velocity_kind="unetx",
    y_is_image=True,
    y_channels=3,
    y_height=28,
    y_width=28,
    num_classes=2,
    x_dim=1,
    unet_c=PLUGIN_UNET_C,
    film_encoder=False,
)
plugin = ConditionalFlowFM(plugin_cfg, device=DEVICE)

trainer.fit_outcome(
    plugin,
    iterations=PLUGIN_STEPS,
    batch_size=PLUGIN_BATCH_SIZE,
    lr=PLUGIN_LR,
    amp=USE_AMP,
    verbose=True,
    log_every=1_000,
)


### Quick nuisance diagnostic

For a few fixed $X$ values, the nuisance should produce the correct digit class while varying the red/blue foreground colour. This is a useful sanity check before paying for target training.

In [ ]:
@torch.no_grad()
def show_image_rows(rows, row_labels, ncols=None, figsize=None):
    nrows = len(rows)
    ncols = len(rows[0]) if ncols is None else ncols
    if figsize is None:
        figsize = (1.55 * ncols, 1.65 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for r, images in enumerate(rows):
        for c in range(ncols):
            axes[r, c].imshow(images[c].permute(1, 2, 0).clamp(0, 1).cpu())
            axes[r, c].axis("off")
            if c == 0:
                axes[r, c].set_title(row_labels[r], loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

x_grid = torch.linspace(0.05, 0.95, 8, device=DEVICE).unsqueeze(1)
with torch.no_grad():
    y_plugin0 = plugin.sample_conditional(x_grid, torch.zeros(8, device=DEVICE), ode_steps=PLUGIN_ODE_STEPS)
    y_plugin1 = plugin.sample_conditional(x_grid, torch.ones(8, device=DEVICE), ode_steps=PLUGIN_ODE_STEPS)
show_image_rows([y_plugin0, y_plugin1], ["plugin: A=0", "plugin: A=1"])


## 4. Train population DeconfoundingFM targets

All three target estimators reuse the **same trained nuisance** and oracle propensity:

1. **DeconfoundingFM** — observational source, independent coupling;
2. **OT-DeconfoundingFM** — observational source, minibatch entropic-OT coupling;
3. **Gaussian-base FM** — matched debiased target objective, but from $N(0,I)$.

For empirical-source targets, every optimizer step gets fresh $X,A,Y$ and fresh draws from $P(Y\mid A=a)$. The plugin draw for each fresh $X$ is selected from the nearest renewable cached context described above.

In [ ]:
population_target_cfg = PopulationTargetConfig(
    context_reservoir_size=CONTEXT_RESERVOIR_SIZE,
    context_refresh_steps=CONTEXT_REFRESH_STEPS,
    plugin_context_chunk=PLUGIN_CONTEXT_CHUNK,
    plugin_ode_steps=PLUGIN_ODE_STEPS,
    reservoir_dtype=RESERVOIR_DTYPE,
    amp=USE_AMP,
)

# Start every target from exactly the same U-Net weights.
torch.manual_seed(SEED + 101)
_velocity_template = UNet(3, 3, num_classes=2, c=TARGET_UNET_C).to(DEVICE)
_initial_velocity_state = copy.deepcopy(_velocity_template.state_dict())
del _velocity_template


def make_target(*, coupling: str, base_kind: str):
    cfg = DeconfoundingFlowConfig(
        dim_y=1,
        base_kind=base_kind,
        batch_size=TARGET_BATCH_SIZE,
        lr=TARGET_LR,
        iterations=None,  # PopulationFlowTrainer owns the exact step budget.
        ode_steps=TARGET_ODE_STEPS,
        min_propensity=1e-2,
        plugin_reservoir=PLUGIN_RESERVOIR,
        plugin_batch=PLUGIN_BATCH,
        base_noise_std=BASE_NOISE_STD if base_kind == "empirical" else 0.0,
        use_ot=(coupling == "ot"),
        ot_iters=20,
        ot_src_batch=128,
        ot_plugin_batch=1,
        ot_eps_scale=0.1,
    )
    velocity = UNet(3, 3, num_classes=2, c=TARGET_UNET_C).to(DEVICE)
    velocity.load_state_dict(_initial_velocity_state)
    return DeconfoundingFlow(
        cfg,
        nuisance_outcome=plugin,
        nuisance_pi=propensity,
        device=DEVICE,
        velocity=velocity,
    )

models = {
    "DeconfoundingFM": make_target(coupling="independent", base_kind="empirical"),
    "OT-DeconfoundingFM": make_target(coupling="ot", base_kind="empirical"),
    "Gaussian-base FM": make_target(coupling="independent", base_kind="gaussian"),
}


In [ ]:
CHECKPOINTS = [TARGET_STEPS] if SMOKE else [2_000, 5_000, 10_000, 20_000]

for name, model in models.items():
    print("\n===", name, "===")
    # Reset the top-level RNG before each fit so nuisance-reservoir construction
    # begins from a reproducible stream. OT itself consumes additional randomness.
    torch.manual_seed(SEED + 1000)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 1000)
    trainer.fit_target(
        model,
        iterations=TARGET_STEPS,
        batch_size=TARGET_BATCH_SIZE,
        lr=TARGET_LR,
        population=population_target_cfg,
        checkpoint_steps=CHECKPOINTS,
        verbose=True,
        log_every=1_000,
    )


## 5. Evaluate against fresh interventional population samples

SW2 is computed in flattened RGB image space using a fixed set of random projections. The reference samples are newly drawn from the true interventional DGP rather than taken from a finite held-out dataset.

In [ ]:
@torch.no_grad()
def sliced_w2_images(x, y, *, n_projections=64, seed=12345, device=DEVICE):
    n = min(len(x), len(y))
    x = x[:n].reshape(n, -1).float().to(device)
    y = y[:n].reshape(n, -1).float().to(device)
    generator = torch.Generator(device=device).manual_seed(seed)
    directions = torch.randn(
        n_projections, x.shape[1], generator=generator, device=device
    )
    directions = directions / directions.norm(dim=1, keepdim=True).clamp_min(1e-12)
    px = x @ directions.T
    py = y @ directions.T
    px = torch.sort(px, dim=0).values
    py = torch.sort(py, dim=0).values
    return float(torch.sqrt((px - py).square().mean()).cpu())

references = {}
sources = {}
predictions = {name: {} for name in models}
metrics = {}

for arm in (0, 1):
    references[arm] = source.sample_interventional(arm, N_EVAL)["Y"].cpu()
    sources[arm] = source.sample_source(arm, N_EVAL)["Y"].cpu()
    for name, model in models.items():
        predictions[name][arm] = trainer.sample_target(
            model, arm, N_EVAL, chunk_size=EVAL_CHUNK, ode_steps=TARGET_ODE_STEPS
        )

for name in ["Observed source", *models.keys()]:
    arm_values = []
    for arm in (0, 1):
        candidate = sources[arm] if name == "Observed source" else predictions[name][arm]
        arm_values.append(
            sliced_w2_images(candidate, references[arm], n_projections=N_PROJECTIONS)
        )
    metrics[name] = arm_values

print(f"{'method':<24} {'SW2 arm 0':>12} {'SW2 arm 1':>12} {'mean':>12}")
for name, values in metrics.items():
    print(f"{name:<24} {values[0]:12.4f} {values[1]:12.4f} {np.mean(values):12.4f}")


In [ ]:
# Visual comparison for do(A=1).
n_show = 8
rows = [
    references[1][:n_show],
    sources[1][:n_show],
    predictions["DeconfoundingFM"][1][:n_show],
    predictions["OT-DeconfoundingFM"][1][:n_show],
    predictions["Gaussian-base FM"][1][:n_show],
]
show_image_rows(
    rows,
    ["true P(Y(1))", "observed P(Y|A=1)", "DeconfoundingFM", "OT-DeconfoundingFM", "Gaussian-base FM"],
    figsize=(12.5, 8.0),
)


## 6. Did the learned generators recover the balanced colour population?

Because the foreground colour encodes $X$, we can approximately recover $X$ from each generated image using $R/(R+B)$ on foreground pixels. Under the intervention, both arms should recover the same uniform colour distribution.

In [ ]:
@torch.no_grad()
def recover_color_x(images, threshold=0.1, eps=1e-8):
    y = images.float().clamp(0, 1)
    r, b = y[:, 0], y[:, 2]
    mask = torch.maximum(r, b) >= threshold
    denom = mask.sum(dim=(1, 2)).clamp_min(1)
    r_mean = (r * mask).sum(dim=(1, 2)) / denom
    b_mean = (b * mask).sum(dim=(1, 2)) / denom
    return r_mean / (r_mean + b_mean + eps)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4), sharex=True, sharey=True)
bins = np.linspace(0, 1, 31)
for ax, (name, by_arm) in zip(axes, predictions.items()):
    for arm, label in [(0, "do(A=0)"), (1, "do(A=1)")]:
        xhat = recover_color_x(by_arm[arm]).numpy()
        ax.hist(xhat, bins=bins, density=True, histtype="step", linewidth=1.8, label=label)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.3, label="true P(X)")
    ax.set_title(name)
    ax.set_xlabel("recovered X = R/(R+B)")
axes[0].set_ylabel("density")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()


## Takeaway

This notebook deliberately removes finite observational-sample reuse from the ColorMNIST experiment. The nuisance learns from fresh $(X,A,Y)$ minibatches, while target training redraws observed outcomes and observational source images at every update. The renewable plugin context reservoir isolates the remaining approximation: each fresh $X$ uses the nearest cached context to amortize expensive conditional-flow ODE samples, and that cache is periodically refreshed.

For a stricter approximation to the population objective, decrease `CONTEXT_REFRESH_STEPS` and/or increase `CONTEXT_RESERVOIR_SIZE`; for a faster run, do the reverse or reduce `PLUGIN_RESERVOIR`.